# ROBO CONDUCTOR：8音パッドできらきら星

このノートブック**1つだけ**で実行できます。上から順番にセルを実行してください。追加のZIP・Pythonファイル・APIキーは、きらきら星の再生だけなら不要です。

- 初期設定：ド〜高いドの8パッドで「きらきら星」の主旋律を演奏。約78.6秒、37.5 BPM、640×360の確認用動画を生成します。描画には時間がかかります。1920×1080は大幅に時間がかかり、使用メモリや保存容量も増えます。
- 元の演奏は**手動で用意した旋律**であり、AI生成ではありません。課題のAI利用を示すには、下の任意のAI計画モードとログも確認してください。
- 色の変化は画面上の演出、音はPyBulletの接触時刻を基に後から合成した電子音です。物理的な発光や衝突音ではありません。
- Mac版は検証済みですが、**このノートブックのGoogle Colab上での実行は未検証**です。ランタイム接続が切れると一時ファイルは消えるので、最後に保存してください。

## 1. ライブラリの準備
Colabでwheelを入手できるよう、Apple Silicon以外にも対応する非公式フォーク `pybullet-arm64==3.2.8` を使用します。公式Bulletとは別の配布です。CPUで動作し、GPUやOpenAI APIキーは初期設定では不要です。


In [ ]:
import importlib.metadata as metadata
import importlib.util
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0], "OS:", platform.system(), platform.machine())
required = ("numpy", "imageio", "imageio_ffmpeg", "PIL", "pybullet")
try:
    fork_version = metadata.version("pybullet-arm64")
except metadata.PackageNotFoundError:
    fork_version = None
try:
    upstream_version = metadata.version("pybullet")
except metadata.PackageNotFoundError:
    upstream_version = None
if upstream_version and fork_version != "3.2.8":
    raise RuntimeError("別のpybulletが入っています。競合を避けるため、Colabの『ランタイムを接続解除して削除』で新しいランタイムを使ってください。")
if fork_version != "3.2.8" or not all(importlib.util.find_spec(name) for name in required):
    packages = ["pybullet-arm64==3.2.8", "numpy>=1.24,<3", "imageio>=2.31,<3", "imageio-ffmpeg>=0.4,<1", "Pillow>=9,<14"]
    print("必要なライブラリをインストールします。")
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "--only-binary=:all:", *packages], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError("対応する配布ファイルが見つからないか、インストールに失敗しました。Colabの標準ランタイムとネットワークを確認してください。Python/OSに対応するwheelがない場合はこの方法では実行できません。") from exc
else:
    print("必要なライブラリは利用できます。")


## 2. 演奏と画質を選択

通常は初期設定のままで大丈夫です。録画時間は「きらきら星」単独の場合です。AI計画を先に加えるとその分長くなります。

In [ ]:
#@title 演奏と動画の設定 { display-mode: "form" }
演奏 = "きらきら星" #@param ["きらきら星", "ドレミ音階"]
テンポ = "約78.6秒（37.5 BPM）" #@param ["約78.6秒（37.5 BPM）", "約59.4秒（50 BPM）"]
画質 = "確認用（640×360）" #@param ["軽量（320×180）", "確認用（640×360）", "フルHD（1920×1080）"]
AI計画 = "使わない" #@param ["使わない", "保存済みの計画を先に演奏", "新しくAPIで計画して先に演奏"]
指示 = "赤、青、緑、赤の順番で叩いて" #@param {type:"string"}
# 「新しくAPI」を選んだ場合のみ、APIへ指示を送信します。ColabのSecretsに
# OPENAI_API_KEYを設定し、このノートブックへのアクセスを許可してください。
# APIの利用には料金が発生する場合があります。キーをこのセルに直接書かないでください。


## 3. 実行用フォルダを作成

In [ ]:
from pathlib import Path
import os
import sys
import tempfile

# 実行ごとに新しいフォルダを作り、以前のファイルを上書きしません。
実行フォルダ = Path(tempfile.mkdtemp(prefix="robo_conductor_", dir=str(Path.cwd())))
os.chdir(実行フォルダ)
sys.path.insert(0, str(実行フォルダ))
for _name in ("melody_normal", "melody_fast", "robo_conductor_baseline"):
    sys.modules.pop(_name, None)
print("保存先:", 実行フォルダ)


## 4. プログラムを準備

このセルには従来の演奏コードが含まれています。

In [ ]:
%%writefile melody_normal.py
"""Eight-pad contact-driven melody, independent of the original three-pad demo.

Twinkle Twinkle Little Star is the traditional melody arranged here as six
phrases. Its rhythm is one beat per note, two beats at each phrase end.
"""
from __future__ import annotations
import argparse
import json
import math
import subprocess
import tempfile
import wave
from pathlib import Path

import imageio.v2 as imageio
import imageio_ffmpeg
import numpy as np
import pybullet as p
import pybullet_data
from PIL import Image, ImageDraw, ImageFont

HZ = 240
NOTES = ('C4', 'D4', 'E4', 'F4', 'G4', 'A4', 'B4', 'C5')
SOLFEGE = dict(zip(NOTES, ('Do', 'Re', 'Mi', 'Fa', 'Sol', 'La', 'Si', 'Do+')))
FREQ = dict(zip(NOTES, (261.625565, 293.664768, 329.627557, 349.228231,
                        391.995436, 440.0, 493.883301, 523.251131)))
COLORS = dict(zip(NOTES, ((0.85,0.08,0.08,1), (0.95,0.35,0.05,1),
                           (0.88,0.75,0.04,1), (0.15,0.68,0.12,1),
                           (0.02,0.58,0.7,1), (0.1,0.25,0.85,1),
                           (0.55,0.14,0.77,1), (0.8,0.18,0.48,1))))
# Two rows of four. Centers are 15 cm apart; pads are 11 cm wide.
POSITIONS = {n:(x,y,0.055) for n,(x,y) in zip(NOTES,
    ((0.42,-0.225),(0.42,-0.075),(0.42,0.075),(0.42,0.225),
     (0.57,-0.225),(0.57,-0.075),(0.57,0.075),(0.57,0.225)))}
PHRASES = (
    ('C4','C4','G4','G4','A4','A4','G4'),
    ('F4','F4','E4','E4','D4','D4','C4'),
    ('G4','G4','F4','F4','E4','E4','D4'),
    ('G4','G4','F4','F4','E4','E4','D4'),
    ('C4','C4','G4','G4','A4','A4','G4'),
    ('F4','F4','E4','E4','D4','D4','C4'),
)


def make_score(piece):
    if piece == 'scale':
        return [{'note': note, 'beats': 1, 'phrase': 'scale'} for note in NOTES]
    if piece != 'twinkle':
        raise ValueError('Unknown piece')
    return [{'note': note, 'beats': 2 if i == len(phrase)-1 else 1,
             'phrase': phrase_index+1}
            for phrase_index, phrase in enumerate(PHRASES)
            for i,note in enumerate(phrase)]


def synthesize(contacts, duration, beat_seconds, path, rate=48000):
    samples = np.zeros(math.ceil(duration * rate), dtype=np.float64)
    for event in contacts:
        # Wrong-pad contacts also make the sound of the actual pad.
        beats = event.get('beats', 1)
        length = int(min(beat_seconds * beats * 0.88, duration) * rate)
        start = round(event['time_s'] * rate)
        end = min(len(samples), start + length)
        length = end-start
        if length <= 0:
            continue
        t = np.arange(length) / rate
        env = np.minimum(1, t / 0.012) * np.minimum(1, (length/rate-t) / 0.10)
        env *= np.exp(-t*0.5)
        f = FREQ[event['note']]
        tone = (np.sin(2*np.pi*f*t) + 0.22*np.sin(2*np.pi*2*f*t) +
                0.07*np.sin(2*np.pi*3*f*t)) / 1.29
        samples[start:end] += 0.30 * env * tone
    pcm = (np.clip(samples, -0.95, 0.95) * 32767).astype('<i2')
    with wave.open(str(path), 'wb') as f:
        f.setnchannels(1); f.setsampwidth(2); f.setframerate(rate)
        f.writeframes(pcm.tobytes())


def _font(size):
    for path in ('/System/Library/Fonts/Helvetica.ttc',
                 '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'):
        try: return ImageFont.truetype(path,size)
        except OSError: pass
    return ImageFont.load_default()


def render(width,height,client,active,piece):
    view=p.computeViewMatrix(cameraEyePosition=[1.18,-1.1,1.25],
                             cameraTargetPosition=[0.34,0,0.28],cameraUpVector=[0,0,1])
    projection=p.computeProjectionMatrixFOV(fov=55,aspect=width/height,nearVal=.05,farVal=5)
    rgba=p.getCameraImage(width,height,viewMatrix=view,projectionMatrix=projection,
                          renderer=p.ER_TINY_RENDERER,physicsClientId=client)[2]
    img=Image.fromarray(np.asarray(rgba,dtype=np.uint8).reshape(height,width,4)[:,:,:3])
    draw=ImageDraw.Draw(img,'RGBA'); sz=max(10,round(height*.034)); font=_font(sz)
    title=_font(max(12,round(height*.045))); margin=max(8,round(height*.025))
    draw.text((margin,margin), 'TWINKLE TWINKLE LITTLE STAR' if piece=='twinkle' else 'DO RE MI - PAD CHECK', font=title, fill=(30,45,65,255))
    y=margin+round(height*.065); line=max(sz+3,round(height*.042))
    for note in NOTES:
        rgb=tuple(round(x*255) for x in COLORS[note][:3]); selected=active == note
        draw.rectangle((margin,y+2,margin+line//2,y+line//2+2),fill=(*rgb,255),outline=(20,20,20,255))
        draw.text((margin+line*.7,y),f'{SOLFEGE[note]}  {note}'+('  *' if selected else ''),font=font,fill=(15,20,30,255))
        y+=line
    return np.asarray(img)


def run(score, piece, output, log, audio, fps=30, width=640, height=360):
    if fps <= 0 or HZ % fps or width < 64 or height < 64 or width%2 or height%2:
        raise ValueError('fps must divide 240; width/height must be even and >=64')
    output, log, audio = map(Path,(output,log,audio))
    if output.suffix.lower() != '.mp4' or audio.suffix.lower() != '.wav':
        raise ValueError('output must be MP4 and audio WAV')
    if len({output.resolve(),log.resolve(),audio.resolve()}) != 3:
        raise ValueError('output, audio and log must be distinct')
    for path in (output,log,audio): path.parent.mkdir(parents=True,exist_ok=True)
    if any(e.get('note') not in NOTES or e.get('beats') not in (1,2) for e in score):
        raise ValueError('Invalid score')
    # Four phase durations add to 1.6 seconds at 30 fps. All are rounded to
    # whole frames, keeping simulation and video clocks aligned for any fps.
    phase_seconds=(.6,.4,.2,.4)
    phase_frames=[max(1,round(s*fps)) for s in phase_seconds]
    beat_frames=sum(phase_frames); beat_seconds=beat_frames/fps
    steps_per_frame=HZ//fps
    client=p.connect(p.DIRECT)
    if client < 0: raise RuntimeError('Could not connect to PyBullet')
    writer=None; frames=0; steps=0; current=None; strikes=[]; contacts=[]
    touching={n:False for n in NOTES}; glow_until={n:-1 for n in NOTES}
    try:
        p.setAdditionalSearchPath(pybullet_data.getDataPath(),physicsClientId=client)
        p.setTimeStep(1/HZ,physicsClientId=client);p.setGravity(0,0,-9.81,physicsClientId=client)
        p.loadURDF('plane.urdf',physicsClientId=client)
        robot=p.loadURDF('kuka_iiwa/model.urdf',[0,0,0],useFixedBase=True,physicsClientId=client)
        col=p.createCollisionShape(p.GEOM_BOX,halfExtents=[.055,.055,.012],physicsClientId=client)
        pads={}
        for note in NOTES:
            vis=p.createVisualShape(p.GEOM_BOX,halfExtents=[.055,.055,.012],rgbaColor=COLORS[note],physicsClientId=client)
            pads[note]=p.createMultiBody(baseMass=0,baseCollisionShapeIndex=col,baseVisualShapeIndex=vis,
                                         basePosition=POSITIONS[note],physicsClientId=client)
        joints=[i for i in range(p.getNumJoints(robot,physicsClientId=client))
                if p.getJointInfo(robot,i,physicsClientId=client)[2]==p.JOINT_REVOLUTE]
        if len(joints)!=7: raise RuntimeError('Unexpected KUKA joint layout')
        ee=6; home=[.36,0,.65]
        with tempfile.TemporaryDirectory(prefix='robo-melody-') as tmp:
            silent=str(Path(tmp)/'silent.mp4')
            writer=imageio.get_writer(silent,fps=fps,codec='libx264',quality=7,macro_block_size=1)
            def move(goal,nframes):
                nonlocal frames,steps,current
                count=nframes*steps_per_frame
                start=np.array(p.getLinkState(robot,ee,computeForwardKinematics=True,physicsClientId=client)[4]);dest=np.array(goal)
                for n in range(count):
                    if steps%steps_per_frame==0:
                        for note,body in pads.items():
                            rgba=COLORS[note]
                            if steps < glow_until[note]: rgba=tuple(.65+.35*v for v in rgba[:3])+(1,)
                            p.changeVisualShape(body,-1,rgbaColor=rgba,physicsClientId=client)
                        active=(current['note'] if current and current['contact_confirmed'] and
                                steps < glow_until[current['note']] else None)
                        writer.append_data(render(width,height,client,active,piece));frames+=1
                    alpha=(n+1)/count; smooth=alpha*alpha*(3-2*alpha)
                    target=(start*(1-smooth)+dest*smooth).tolist()
                    qs=p.calculateInverseKinematics(robot,ee,target,maxNumIterations=100,residualThreshold=1e-4,physicsClientId=client)
                    for j,q in zip(joints,qs):
                        p.setJointMotorControl2(robot,j,p.POSITION_CONTROL,targetPosition=float(q),force=250,physicsClientId=client)
                    p.stepSimulation(physicsClientId=client);steps+=1
                    for note,body in pads.items():
                        points=p.getContactPoints(bodyA=robot,bodyB=body,linkIndexA=ee,physicsClientId=client)
                        points=[pt for pt in points if pt[9]>0]
                        now=bool(points)
                        if now and not touching[note]:
                            pt=max(points,key=lambda v:v[9])
                            event={'note':note,'time_s':steps/HZ,'robot_link':ee,'position_on_pad':list(pt[6]),
                                   'normal_force_n':float(pt[9]),'strike':current['strike'] if current else None,
                                   'intended_note':current['note'] if current else None,
                                   'beats':current['beats'] if current and current['note']==note else 1}
                            contacts.append(event);glow_until[note]=steps+round(.25*HZ)
                            if current and current['note']==note and not current['contact_confirmed']:
                                current.update(contact_confirmed=True,contact_time_s=steps/HZ,
                                               normal_force_n=event['normal_force_n'],position_on_pad=event['position_on_pad'])
                        touching[note]=now
            move(home,round(1.2*fps))
            for i,item in enumerate(score,1):
                note=item['note'];x,y,z=POSITIONS[note]
                current={'strike':i,'note':note,'beats':item['beats'],'phrase':item.get('phrase'),
                         'start_time_s':steps/HZ,'contact_confirmed':False,'contact_time_s':None,'normal_force_n':None}
                strikes.append(current)
                above=[x,y,z+.25]; hit=[x,y,z+.02]
                move(above,phase_frames[0]);move(hit,phase_frames[1]);move(hit,phase_frames[2]);move(above,phase_frames[3])
                for _ in range(item['beats']-1): move(above,beat_frames)
                current['end_time_s']=steps/HZ;current=None
            move(home,round(.6*fps))
            writer.close();writer=None
            if frames*steps_per_frame != steps: raise RuntimeError('video and physics clocks differ')
            duration=frames/fps
            synthesize(contacts,duration,beat_seconds,audio)
            subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(),'-y','-loglevel','error','-i',silent,'-i',str(audio),
                            '-c:v','copy','-c:a','aac','-b:a','192k','-movflags','+faststart','-shortest',str(output)],check=True)
        data={'piece':piece,'source':'hand-authored traditional melody' if piece=='twinkle' else 'scale check',
              'video':str(output),'audio':str(audio),'fps':fps,'width':width,'height':height,'duration_s':duration,
              'frame_count':frames,'simulation_hz':HZ,'nominal_beat_s':beat_seconds,'nominal_bpm':60/beat_seconds,
              'events':strikes,'contacts':contacts,'successful_strikes':sum(e['contact_confirmed'] for e in strikes),
              'total_strikes':len(strikes),
              'notes':['Sound is synthesized from actual force-bearing contact onsets, including unintended pads.',
                       'Pad glow is a color change; audio is not a physical impact sound.',
                       'Beat timing is nominal; actual contact onsets can differ due to robot movement.']}
        log.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8')
        return data
    finally:
        if writer: writer.close()
        p.disconnect(physicsClientId=client)


def main():
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--piece',choices=('scale','twinkle'),default='twinkle')
    parser.add_argument('--output',default='output/twinkle.mp4');parser.add_argument('--log',default='output/twinkle.json')
    parser.add_argument('--audio',default='output/twinkle.wav');parser.add_argument('--fps',type=int,default=30)
    parser.add_argument('--width',type=int,default=640);parser.add_argument('--height',type=int,default=360)
    args=parser.parse_args()
    r=run(make_score(args.piece),args.piece,args.output,args.log,args.audio,args.fps,args.width,args.height)
    print(f"{r['piece']}: {r['successful_strikes']}/{r['total_strikes']} contacts; {r['duration_s']:.2f}s; {r['nominal_bpm']:.2f} BPM")

if __name__=='__main__':main()


In [ ]:
%%writefile melody_fast.py
"""Eight-pad contact-driven melody, independent of the original three-pad demo.

Twinkle Twinkle Little Star is the traditional melody arranged here as six
phrases. Its rhythm is one beat per note, two beats at each phrase end.
"""
from __future__ import annotations
import argparse
import json
import math
import subprocess
import tempfile
import wave
from pathlib import Path

import imageio.v2 as imageio
import imageio_ffmpeg
import numpy as np
import pybullet as p
import pybullet_data
from PIL import Image, ImageDraw, ImageFont

HZ = 240
NOTES = ('C4', 'D4', 'E4', 'F4', 'G4', 'A4', 'B4', 'C5')
SOLFEGE = dict(zip(NOTES, ('Do', 'Re', 'Mi', 'Fa', 'Sol', 'La', 'Si', 'Do+')))
FREQ = dict(zip(NOTES, (261.625565, 293.664768, 329.627557, 349.228231,
                        391.995436, 440.0, 493.883301, 523.251131)))
COLORS = dict(zip(NOTES, ((0.85,0.08,0.08,1), (0.95,0.35,0.05,1),
                           (0.88,0.75,0.04,1), (0.15,0.68,0.12,1),
                           (0.02,0.58,0.7,1), (0.1,0.25,0.85,1),
                           (0.55,0.14,0.77,1), (0.8,0.18,0.48,1))))
# Two rows of four. Centers are 15 cm apart; pads are 11 cm wide.
POSITIONS = {n:(x,y,0.055) for n,(x,y) in zip(NOTES,
    ((0.42,-0.225),(0.42,-0.075),(0.42,0.075),(0.42,0.225),
     (0.57,-0.225),(0.57,-0.075),(0.57,0.075),(0.57,0.225)))}
PHRASES = (
    ('C4','C4','G4','G4','A4','A4','G4'),
    ('F4','F4','E4','E4','D4','D4','C4'),
    ('G4','G4','F4','F4','E4','E4','D4'),
    ('G4','G4','F4','F4','E4','E4','D4'),
    ('C4','C4','G4','G4','A4','A4','G4'),
    ('F4','F4','E4','E4','D4','D4','C4'),
)


def make_score(piece):
    if piece == 'scale':
        return [{'note': note, 'beats': 1, 'phrase': 'scale'} for note in NOTES]
    if piece != 'twinkle':
        raise ValueError('Unknown piece')
    return [{'note': note, 'beats': 2 if i == len(phrase)-1 else 1,
             'phrase': phrase_index+1}
            for phrase_index, phrase in enumerate(PHRASES)
            for i,note in enumerate(phrase)]


def synthesize(contacts, duration, beat_seconds, path, rate=48000):
    samples = np.zeros(math.ceil(duration * rate), dtype=np.float64)
    for event in contacts:
        # Wrong-pad contacts also make the sound of the actual pad.
        beats = event.get('beats', 1)
        length = int(min(beat_seconds * beats * 0.88, duration) * rate)
        start = round(event['time_s'] * rate)
        end = min(len(samples), start + length)
        length = end-start
        if length <= 0:
            continue
        t = np.arange(length) / rate
        env = np.minimum(1, t / 0.012) * np.minimum(1, (length/rate-t) / 0.10)
        env *= np.exp(-t*0.5)
        f = FREQ[event['note']]
        tone = (np.sin(2*np.pi*f*t) + 0.22*np.sin(2*np.pi*2*f*t) +
                0.07*np.sin(2*np.pi*3*f*t)) / 1.29
        samples[start:end] += 0.30 * env * tone
    pcm = (np.clip(samples, -0.95, 0.95) * 32767).astype('<i2')
    with wave.open(str(path), 'wb') as f:
        f.setnchannels(1); f.setsampwidth(2); f.setframerate(rate)
        f.writeframes(pcm.tobytes())


def _font(size):
    for path in ('/System/Library/Fonts/Helvetica.ttc',
                 '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'):
        try: return ImageFont.truetype(path,size)
        except OSError: pass
    return ImageFont.load_default()


def render(width,height,client,active,piece):
    view=p.computeViewMatrix(cameraEyePosition=[1.18,-1.1,1.25],
                             cameraTargetPosition=[0.34,0,0.28],cameraUpVector=[0,0,1])
    projection=p.computeProjectionMatrixFOV(fov=55,aspect=width/height,nearVal=.05,farVal=5)
    rgba=p.getCameraImage(width,height,viewMatrix=view,projectionMatrix=projection,
                          renderer=p.ER_TINY_RENDERER,physicsClientId=client)[2]
    img=Image.fromarray(np.asarray(rgba,dtype=np.uint8).reshape(height,width,4)[:,:,:3])
    draw=ImageDraw.Draw(img,'RGBA'); sz=max(10,round(height*.034)); font=_font(sz)
    title=_font(max(12,round(height*.045))); margin=max(8,round(height*.025))
    draw.text((margin,margin), 'TWINKLE TWINKLE LITTLE STAR' if piece=='twinkle' else 'DO RE MI - PAD CHECK', font=title, fill=(30,45,65,255))
    y=margin+round(height*.065); line=max(sz+3,round(height*.042))
    for note in NOTES:
        rgb=tuple(round(x*255) for x in COLORS[note][:3]); selected=active == note
        draw.rectangle((margin,y+2,margin+line//2,y+line//2+2),fill=(*rgb,255),outline=(20,20,20,255))
        draw.text((margin+line*.7,y),f'{SOLFEGE[note]}  {note}'+('  *' if selected else ''),font=font,fill=(15,20,30,255))
        y+=line
    return np.asarray(img)


def run(score, piece, output, log, audio, fps=30, width=640, height=360):
    if fps <= 0 or HZ % fps or width < 64 or height < 64 or width%2 or height%2:
        raise ValueError('fps must divide 240; width/height must be even and >=64')
    output, log, audio = map(Path,(output,log,audio))
    if output.suffix.lower() != '.mp4' or audio.suffix.lower() != '.wav':
        raise ValueError('output must be MP4 and audio WAV')
    if len({output.resolve(),log.resolve(),audio.resolve()}) != 3:
        raise ValueError('output, audio and log must be distinct')
    for path in (output,log,audio): path.parent.mkdir(parents=True,exist_ok=True)
    if any(e.get('note') not in NOTES or e.get('beats') not in (1,2) for e in score):
        raise ValueError('Invalid score')
    # Trial only: these phase durations sum to 1.2 seconds at 30 fps. Rounded to
    # whole frames, keeping simulation and video clocks aligned for any fps.
    phase_seconds=(.45,.3,.15,.3)
    phase_frames=[max(1,round(s*fps)) for s in phase_seconds]
    beat_frames=sum(phase_frames); beat_seconds=beat_frames/fps
    steps_per_frame=HZ//fps
    client=p.connect(p.DIRECT)
    if client < 0: raise RuntimeError('Could not connect to PyBullet')
    writer=None; frames=0; steps=0; current=None; strikes=[]; contacts=[]
    touching={n:False for n in NOTES}; glow_until={n:-1 for n in NOTES}
    try:
        p.setAdditionalSearchPath(pybullet_data.getDataPath(),physicsClientId=client)
        p.setTimeStep(1/HZ,physicsClientId=client);p.setGravity(0,0,-9.81,physicsClientId=client)
        p.loadURDF('plane.urdf',physicsClientId=client)
        robot=p.loadURDF('kuka_iiwa/model.urdf',[0,0,0],useFixedBase=True,physicsClientId=client)
        col=p.createCollisionShape(p.GEOM_BOX,halfExtents=[.055,.055,.012],physicsClientId=client)
        pads={}
        for note in NOTES:
            vis=p.createVisualShape(p.GEOM_BOX,halfExtents=[.055,.055,.012],rgbaColor=COLORS[note],physicsClientId=client)
            pads[note]=p.createMultiBody(baseMass=0,baseCollisionShapeIndex=col,baseVisualShapeIndex=vis,
                                         basePosition=POSITIONS[note],physicsClientId=client)
        joints=[i for i in range(p.getNumJoints(robot,physicsClientId=client))
                if p.getJointInfo(robot,i,physicsClientId=client)[2]==p.JOINT_REVOLUTE]
        if len(joints)!=7: raise RuntimeError('Unexpected KUKA joint layout')
        ee=6; home=[.36,0,.65]
        with tempfile.TemporaryDirectory(prefix='robo-melody-') as tmp:
            silent=str(Path(tmp)/'silent.mp4')
            writer=imageio.get_writer(silent,fps=fps,codec='libx264',quality=7,macro_block_size=1)
            def move(goal,nframes):
                nonlocal frames,steps,current
                count=nframes*steps_per_frame
                start=np.array(p.getLinkState(robot,ee,computeForwardKinematics=True,physicsClientId=client)[4]);dest=np.array(goal)
                for n in range(count):
                    if steps%steps_per_frame==0:
                        for note,body in pads.items():
                            rgba=COLORS[note]
                            if steps < glow_until[note]: rgba=tuple(.65+.35*v for v in rgba[:3])+(1,)
                            p.changeVisualShape(body,-1,rgbaColor=rgba,physicsClientId=client)
                        active=(current['note'] if current and current['contact_confirmed'] and
                                steps < glow_until[current['note']] else None)
                        writer.append_data(render(width,height,client,active,piece));frames+=1
                    alpha=(n+1)/count; smooth=alpha*alpha*(3-2*alpha)
                    target=(start*(1-smooth)+dest*smooth).tolist()
                    qs=p.calculateInverseKinematics(robot,ee,target,maxNumIterations=100,residualThreshold=1e-4,physicsClientId=client)
                    for j,q in zip(joints,qs):
                        p.setJointMotorControl2(robot,j,p.POSITION_CONTROL,targetPosition=float(q),force=250,physicsClientId=client)
                    p.stepSimulation(physicsClientId=client);steps+=1
                    for note,body in pads.items():
                        points=p.getContactPoints(bodyA=robot,bodyB=body,linkIndexA=ee,physicsClientId=client)
                        points=[pt for pt in points if pt[9]>0]
                        now=bool(points)
                        if now and not touching[note]:
                            pt=max(points,key=lambda v:v[9])
                            event={'note':note,'time_s':steps/HZ,'robot_link':ee,'position_on_pad':list(pt[6]),
                                   'normal_force_n':float(pt[9]),'strike':current['strike'] if current else None,
                                   'intended_note':current['note'] if current else None,
                                   'beats':current['beats'] if current and current['note']==note else 1}
                            contacts.append(event);glow_until[note]=steps+round(.25*HZ)
                            if current and current['note']==note and not current['contact_confirmed']:
                                current.update(contact_confirmed=True,contact_time_s=steps/HZ,
                                               normal_force_n=event['normal_force_n'],position_on_pad=event['position_on_pad'])
                        touching[note]=now
            move(home,round(1.2*fps))
            for i,item in enumerate(score,1):
                note=item['note'];x,y,z=POSITIONS[note]
                current={'strike':i,'note':note,'beats':item['beats'],'phrase':item.get('phrase'),
                         'start_time_s':steps/HZ,'contact_confirmed':False,'contact_time_s':None,'normal_force_n':None}
                strikes.append(current)
                above=[x,y,z+.25]; hit=[x,y,z+.02]
                move(above,phase_frames[0]);move(hit,phase_frames[1]);move(hit,phase_frames[2]);move(above,phase_frames[3])
                for _ in range(item['beats']-1): move(above,beat_frames)
                current['end_time_s']=steps/HZ;current=None
            move(home,round(.6*fps))
            writer.close();writer=None
            if frames*steps_per_frame != steps: raise RuntimeError('video and physics clocks differ')
            duration=frames/fps
            synthesize(contacts,duration,beat_seconds,audio)
            subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(),'-y','-loglevel','error','-i',silent,'-i',str(audio),
                            '-c:v','copy','-c:a','aac','-b:a','192k','-movflags','+faststart','-shortest',str(output)],check=True)
        data={'piece':piece,'source':'hand-authored traditional melody' if piece=='twinkle' else 'scale check',
              'video':str(output),'audio':str(audio),'fps':fps,'width':width,'height':height,'duration_s':duration,
              'frame_count':frames,'simulation_hz':HZ,'nominal_beat_s':beat_seconds,'nominal_bpm':60/beat_seconds,
              'events':strikes,'contacts':contacts,'successful_strikes':sum(e['contact_confirmed'] for e in strikes),
              'total_strikes':len(strikes),
              'notes':['Sound is synthesized from actual force-bearing contact onsets, including unintended pads.',
                       'Pad glow is a color change; audio is not a physical impact sound.',
                       'Beat timing is nominal; actual contact onsets can differ due to robot movement.']}
        log.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8')
        return data
    finally:
        if writer: writer.close()
        p.disconnect(physicsClientId=client)


def main():
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--piece',choices=('scale','twinkle'),default='twinkle')
    parser.add_argument('--output',default='output/twinkle.mp4');parser.add_argument('--log',default='output/twinkle.json')
    parser.add_argument('--audio',default='output/twinkle.wav');parser.add_argument('--fps',type=int,default=30)
    parser.add_argument('--width',type=int,default=640);parser.add_argument('--height',type=int,default=360)
    args=parser.parse_args()
    r=run(make_score(args.piece),args.piece,args.output,args.log,args.audio,args.fps,args.width,args.height)
    print(f"{r['piece']}: {r['successful_strikes']}/{r['total_strikes']} contacts; {r['duration_s']:.2f}s; {r['nominal_bpm']:.2f} BPM")

if __name__=='__main__':main()


元のAI計画用コードを別ファイルとして保持します。通常の演奏ではAPIへ接続しません。

In [ ]:
%%writefile robo_conductor_baseline.py
"""ROBO CONDUCTOR: LLM-planned drum-pad tapping in PyBullet (Colab compatible)."""
import os, json, math, argparse
from pathlib import Path
import numpy as np
import pybullet as p
import pybullet_data
import imageio.v2 as imageio

COLORS = ('red', 'blue', 'green')
PAD_POS = {'red': (0.48, -0.24, 0.055), 'blue': (0.54, 0.0, 0.055), 'green': (0.48, 0.24, 0.055)}
RGB = {'red': (0.9, 0.12, 0.1, 1), 'blue': (0.1, 0.3, 0.9, 1), 'green': (0.1, 0.75, 0.2, 1)}


def make_plan(instruction, model='gpt-4.1-mini'):
    """Calls an actual LLM. Requires OPENAI_API_KEY; no simulated AI fallback."""
    from openai import OpenAI
    key = os.environ.get('OPENAI_API_KEY')
    if not key:
        raise RuntimeError('OPENAI_API_KEY is missing. Set it in Colab Secrets or environment.')
    client = OpenAI(api_key=key)
    response = client.chat.completions.create(
        model=model,
        temperature=0,
        response_format={'type': 'json_object'},
        messages=[
            {'role': 'system', 'content': 'You are a robot percussion planner. Return only JSON of the form {"sequence":["red","blue"]}. Allowed values: red, blue, green. Translate Japanese color names 赤=red 青=blue 緑=green. Respect order and repetition. 1 to 12 strikes. No other keys.'},
            {'role': 'user', 'content': instruction},
        ],
    )
    raw = json.loads(response.choices[0].message.content)
    sequence = raw.get('sequence')
    if not isinstance(sequence, list) or not 1 <= len(sequence) <= 12 or any(c not in COLORS for c in sequence):
        raise ValueError(f'Invalid AI plan: {raw}')
    return {'instruction': instruction, 'sequence': sequence, 'model': model, 'source': 'OpenAI API'}


def create_pad(position, rgba):
    half = [0.085, 0.085, 0.012]
    col = p.createCollisionShape(p.GEOM_BOX, halfExtents=half)
    vis = p.createVisualShape(p.GEOM_BOX, halfExtents=half, rgbaColor=rgba)
    return p.createMultiBody(baseMass=0, baseCollisionShapeIndex=col, baseVisualShapeIndex=vis,
                             basePosition=position)


def camera(width, height):
    view = p.computeViewMatrix(cameraEyePosition=[1.15, -1.05, 1.12],
                               cameraTargetPosition=[0.27, 0, 0.38], cameraUpVector=[0, 0, 1])
    proj = p.computeProjectionMatrixFOV(fov=57, aspect=width/height, nearVal=0.05, farVal=5)
    result = p.getCameraImage(width, height, viewMatrix=view, projectionMatrix=proj,
                              renderer=p.ER_TINY_RENDERER)
    return np.asarray(result[2], dtype=np.uint8).reshape(height, width, 4)[:, :, :3].copy()


def simulate(plan, output='robot_demo.mp4', fps=24, width=960, height=540):
    p.connect(p.DIRECT)
    try:
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        p.setGravity(0, 0, -9.81)
        p.setTimeStep(1/240)
        p.loadURDF('plane.urdf')
        robot = p.loadURDF('kuka_iiwa/model.urdf', [0, 0, 0], useFixedBase=True)
        for color in COLORS:
            create_pad(PAD_POS[color], RGB[color])
        # The KUKA iiwa URDF's last link index is 6.
        ee_link = 6
        joint_ids = [j for j in range(p.getNumJoints(robot)) if p.getJointInfo(robot, j)[2] == p.JOINT_REVOLUTE]
        if len(joint_ids) != 7:
            raise RuntimeError('Unexpected KUKA joint layout')
        writer = imageio.get_writer(output, fps=fps, codec='libx264', quality=7, macro_block_size=16)
        frames = 0
        events = []
        sim_steps_per_frame = 10
        home = [0.36, 0, 0.65]

        def move(target, seconds):
            nonlocal frames
            n = max(1, int(seconds * fps))
            start = np.asarray(p.getLinkState(robot, ee_link, computeForwardKinematics=True)[4])
            goal = np.asarray(target)
            for i in range(n):
                alpha = (i + 1) / n
                smooth = alpha * alpha * (3 - 2 * alpha)
                xyz = (start * (1-smooth) + goal * smooth).tolist()
                joints = p.calculateInverseKinematics(robot, ee_link, xyz,
                                                      maxNumIterations=100, residualThreshold=1e-4)
                for j, q in zip(joint_ids, joints):
                    p.setJointMotorControl2(robot, j, p.POSITION_CONTROL,
                                            targetPosition=float(q), force=250)
                for _ in range(sim_steps_per_frame):
                    p.stepSimulation()
                writer.append_data(camera(width, height))
                frames += 1

        # Settle to an initial raised pose.
        move(home, 1.0)
        for idx, color in enumerate(plan['sequence'], 1):
            x, y, z = PAD_POS[color]
            # Approach above the pad, then lower to near the surface, then retreat.
            above = [x, y, z + 0.25]
            contact = [x, y, z + 0.10]
            move(above, 0.65)
            move(contact, 0.35)
            actual = p.getLinkState(robot, ee_link, computeForwardKinematics=True)[4]
            error = float(np.linalg.norm(np.asarray(actual) - np.asarray(contact)))
            events.append({'strike': idx, 'color': color, 'target_xyz': contact,
                           'actual_xyz': list(actual), 'position_error_m': round(error, 4),
                           'video_time_s': round(frames / fps, 2)})
            move(above, 0.35)
        move(home, 0.8)
        writer.close()
        return {'video': output, 'duration_s': round(frames / fps, 2), 'events': events,
                'note': 'This demo uses end-effector proximity to a pad, not verified contact force.'}
    finally:
        p.disconnect()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--instruction', default='赤、青、緑、赤の順番で叩いて')
    parser.add_argument('--output', default='robot_demo.mp4')
    parser.add_argument('--model', default='gpt-4.1-mini')
    args = parser.parse_args()
    plan = make_plan(args.instruction, args.model)
    Path('plan.json').write_text(json.dumps(plan, ensure_ascii=False, indent=2), encoding='utf-8')
    print('AI plan:', plan)
    result = simulate(plan, args.output)
    Path('results.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Result:', json.dumps(result, ensure_ascii=False, indent=2))

if __name__ == '__main__':
    main()


## 5. AI計画（任意）

保存済み計画は元の実行記録で、今回APIを呼び出した証拠ではありません。新規API実行は任意で、指定した文章が外部へ送信され料金が発生する場合があります。元の `make_plan` を利用します。API実行はこのノートブックでは未検証です。

In [ ]:
#@title 任意：AI計画を用意 { display-mode: "form" }
import importlib.util
import json
import os
import subprocess
import sys

if AI計画 not in ("使わない", "保存済みの計画を先に演奏", "新しくAPIで計画して先に演奏"):
    raise ValueError("AI計画の選択が不正です")
計画 = None
if AI計画 == "保存済みの計画を先に演奏":
    # 保存済みの演奏計画です。今回の新規API呼び出しではありません。
    計画 = {'instruction': '赤、青、緑、赤の順番で叩いて', 'sequence': ['red', 'blue', 'green', 'red'], 'model': 'gpt-4.1-mini', 'source': 'OpenAI API'}
    計画["verification_note"] = "計画ファイルに記録された出所。今回APIは呼び出していません。"
    print("保存済みの演奏計画を使用します。今回のAPI呼び出しはありません。")
elif AI計画 == "新しくAPIで計画して先に演奏":
    # 新規API実行が明示的に選ばれた場合だけ、対応SDKを準備します。
    subprocess.run([sys.executable, "-m", "pip", "install", "openai>=1,<3"], check=True)
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None
    キー = userdata.get("OPENAI_API_KEY") if userdata is not None else os.environ.get("OPENAI_API_KEY")
    if not キー:
        raise RuntimeError("OPENAI_API_KEYがありません。ColabのSecretsに登録しアクセスを許可してください。")
    import robo_conductor_baseline as baseline
    以前のキー = os.environ.get("OPENAI_API_KEY")
    try:
        os.environ["OPENAI_API_KEY"] = キー
        計画 = baseline.make_plan(指示)
    finally:
        if 以前のキー is None:
            os.environ.pop("OPENAI_API_KEY", None)
        else:
            os.environ["OPENAI_API_KEY"] = 以前のキー
        del キー
    計画["verification_note"] = "この実行でOpenAI APIを呼び出した計画です。"
    print("新しいAPI計画を取得しました。")
if 計画 is not None:
    print("指示:", 計画.get("instruction"), "色の順序:", 計画.get("sequence"))
    print("音の対応: 赤→ド(C4)、青→ラ(A4)、緑→ファ(F4)")
else:
    print("AI計画は使用しません。次の曲・音階は手動で設定されたものです。")


## 6. 動画を生成

接触しなかった音は成功として記録されず、発音されません。予定外のパッドに触れた場合はそのパッドの音と記録が残ります。

In [ ]:
#@title シミュレーションと動画の生成
import importlib
import json

if 演奏 not in ("きらきら星", "ドレミ音階"):
    raise ValueError("演奏の選択が不正です")
if テンポ not in ("約78.6秒（37.5 BPM）", "約59.4秒（50 BPM）"):
    raise ValueError("テンポの選択が不正です")
解像度一覧 = {"軽量（320×180）": (320, 180), "確認用（640×360）": (640, 360), "フルHD（1920×1080）": (1920, 1080)}
if 画質 not in 解像度一覧:
    raise ValueError("画質の選択が不正です")
幅, 高さ = 解像度一覧[画質]
曲名 = "twinkle" if 演奏 == "きらきら星" else "scale"
モジュール名 = "melody_normal" if テンポ.startswith("約78.6秒") else "melody_fast"
演奏機能 = importlib.import_module(モジュール名)
楽譜 = 演奏機能.make_score(曲名)
if 計画 is not None:
    色から音 = {"red": "C4", "blue": "A4", "green": "F4"}
    色列 = 計画.get("sequence")
    if not isinstance(色列, list) or not 1 <= len(色列) <= 12 or any(色 not in 色から音 for 色 in 色列):
        raise ValueError("計画の色列は red / blue / green の1〜12個にしてください")
    ai楽譜 = [{"note": 色から音[色], "beats": 1, "phrase": "ai_plan"} for 色 in 色列]
    楽譜 = ai楽譜 + 楽譜
動画 = 実行フォルダ / "robo_conductor.mp4"
音声 = 実行フォルダ / "contacts.wav"
記録 = 実行フォルダ / "results.json"
print(f"{演奏}：{len(楽譜)}打、{幅}×{高さ}で生成します。完了まで待ってください。")
結果 = 演奏機能.run(楽譜, 曲名, 動画, 記録, 音声, fps=30, width=幅, height=高さ)
結果["source"] = "AI計画と手動の演奏を連結" if 計画 else "手動で設定した曲または音階"
結果["ai_mode"] = AI計画
結果["ai_plan"] = 計画
結果["note_mapping"] = {"red": "C4", "blue": "A4", "green": "F4"} if 計画 else None
記録.write_text(json.dumps(結果, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"完成: {結果['duration_s']:.2f}秒 / 接触成功 {結果['successful_strikes']}/{結果['total_strikes']}")
print("対象と違うパッドへの接触:", sum(item['note'] != item['intended_note'] for item in 結果['contacts']), "件")


## 7. 確認してダウンロード

動画、接触音声、実行ログをZIPにまとめます。課題の提出動画（1〜3分）には、作品説明やAIの使い方、検証結果も必要に応じて加え、正式な提出要項を確認してください。

In [ ]:
#@title 動画を見る
from IPython.display import Video, display
print("接触はPyBullet内のシミュレーション結果です。実機の安全性を示すものではありません。")
display(Video(str(動画), embed=True))


In [ ]:
#@title 動画・音声・ログをまとめて保存
import zipfile
まとめ = 実行フォルダ / "robo_conductor_output.zip"
with zipfile.ZipFile(まとめ, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in (動画, 音声, 記録):
        z.write(path, arcname=path.name)
print("保存ファイル:", まとめ)
try:
    from google.colab import files
except ImportError:
    print("Colab以外では上記のファイルを開いてください。")
else:
    files.download(str(まとめ))
